# yfinance Download Determinism Investigation

This notebook isolates the data-download inconsistency seen in the ML predictor grid searches. It tests yfinance directly, outside the model, indicators, and backtester.

The key question is:

> When two independent workers call `yf.download(...)` with the same arguments, do they receive exactly the same data?

The short answer demonstrated below is: raw `Open/High/Low/Close/Volume` is stable in this test, while `Adj Close` can differ by tiny amounts across independent worker processes. Because `auto_adjust=True` uses `Adj Close / Close` to adjust all OHLC columns, those tiny differences propagate into the adjusted OHLC dataset.

## 1. Imports And Test Configuration

The chosen date range matches the shorter diagnostic window. The start date below is the actual Yahoo download start for one representative training window.

In [7]:
from joblib import Parallel, delayed
from IPython.display import display

import hashlib
import inspect
import os
import time

import numpy as np
import pandas as pd
import yfinance as yf
from pandas.testing import assert_frame_equal

SYMBOL = "ECH"
START = "2018-01-01"
END = "2025-03-01"
N_WORKERS = 4
N_CALLS = 8

print("yfinance version:", yf.__version__)
print("yf.download signature:", inspect.signature(yf.download))

yfinance version: 1.2.0
yf.download signature: (tickers, start=None, end=None, actions=False, threads=True, ignore_tz=None, group_by='column', auto_adjust=True, back_adjust=False, repair=False, keepna=False, progress=True, period=None, interval='1d', prepost=False, rounding=False, timeout=10, session=None, multi_level_index=True) -> Optional[pandas.core.frame.DataFrame]


## 2. Helpers

The fingerprint is a strict exact-data hash. If two DataFrames differ by even a tiny floating-point amount, their fingerprints differ. That is intentional here: we first want to detect exact non-determinism, then measure whether the differences matter.

In [4]:
def canonicalize_download(df, columns=None):
    """Return a standardized DataFrame for exact comparisons.

    yfinance may return a MultiIndex column layout even for one ticker.
    We flatten it, keep a known column order, sort the dates, and remove timezone
    metadata from the index so superficial formatting does not affect comparisons.
    """
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)

    if columns is None:
        columns = [c for c in ["Open", "High", "Low", "Close", "Adj Close", "Volume"] if c in out.columns]

    out = out[columns].copy().sort_index()
    out.index = pd.to_datetime(out.index).tz_localize(None)
    return out


def dataframe_fingerprint(df):
    """Compute a strict fingerprint of a DataFrame's columns, index, and values."""
    canonical = canonicalize_download(df)
    hasher = hashlib.sha256()
    hasher.update(str(list(canonical.columns)).encode("utf-8"))
    hasher.update(pd.util.hash_pandas_object(canonical, index=True).values.tobytes())
    return hasher.hexdigest()


def download_once(call_id, auto_adjust, columns=None):
    """Make one yfinance download call and return both metadata and the DataFrame.

    threads=False is used deliberately: this test is not about yfinance's internal
    ticker-threading. It is about independent process-level calls with identical
    arguments.
    """
    df = yf.download(
        SYMBOL,
        start=START,
        end=END,
        auto_adjust=auto_adjust,
        threads=False,
        # threads=True,  # I want to test if threads=True works as long as we drop any 'Adjusted' prices.
        progress=False,
        timeout=30,
    )
    df = canonicalize_download(df, columns=columns)
    return {
        "call_id": call_id,
        "pid": os.getpid(),
        "auto_adjust": auto_adjust,
        "rows": len(df),
        "start": df.index.min(),
        "end": df.index.max(),
        "columns": list(df.columns),
        "fingerprint": dataframe_fingerprint(df),
        "df": df,
    }


def summarize_download_records(records, fingerprint_column="fingerprint"):
    """Summarize worker outputs without displaying the full DataFrames."""
    rows = []
    for record in records:
        rows.append({k: v for k, v in record.items() if k != "df"})
    summary = pd.DataFrame(rows)
    summary["fingerprint_short"] = summary[fingerprint_column].str.slice(0, 12)
    return summary


def compare_to_first(records, columns=None):
    """Compare every returned DataFrame against the first returned DataFrame."""
    base = canonicalize_download(records[0]["df"], columns=columns)
    rows = []
    for record in records[1:]:
        other = canonicalize_download(record["df"], columns=columns)
        common_columns = base.columns.intersection(other.columns)
        common_index = base.index.intersection(other.index)
        diff = (base.loc[common_index, common_columns] - other.loc[common_index, common_columns]).abs()
        pct_diff = (diff / base.loc[common_index, common_columns].abs()).replace([np.inf, -np.inf], np.nan)
        rows.append({
            "base_pid": records[0]["pid"],
            "other_pid": record["pid"],
            "index_matches": base.index.equals(other.index),
            "columns_match": list(base.columns) == list(other.columns),
            # "len(diff)": len(diff),
            "shape(diff)": diff.shape if len(diff) else None,
            # "len(base)": len(base),
            # "len(other)": len(other),
            "shape(other)": other.shape if len(other) else None,
            "max_abs_diff": float(diff.max().max()) if len(diff) else np.nan,
            "differing_cells_gt_0": int((diff > 0).sum().sum()) if len(diff) else 0,
            "differing_cells_gt_1e_5": int((diff > 1e-5).sum().sum()) if len(diff) else 0,
            "max_pct_diff": float(pct_diff.max().max()) if len(pct_diff) else np.nan,
            "differing_cells_pct_gt_1e_7": int((pct_diff > 1e-7).sum().sum()) if len(pct_diff) else 0,
            "differing_columns": list(diff.columns[(diff > 0).any()]) if len(diff) else None,
            "max_diff_column": diff.max().idxmax() if len(diff) and int((diff > 1e-8).sum().sum()) else None,
        })
    return pd.DataFrame(rows)

## 3. Same Process Repeated Calls

In one Python process, repeated calls are stable. This is useful but not sufficient: yfinance can use in-process caching for historical date ranges, so this does not prove that independent network calls are stable.

In [5]:
same_process_records = [download_once(i, auto_adjust=False) for i in range(8)]
same_process_summary = summarize_download_records(same_process_records)
display(same_process_summary[["call_id", "pid", "rows", "start", "end", "fingerprint_short"]])
print("Unique fingerprints in same process:", same_process_summary["fingerprint"].nunique())

,call_id,pid,rows,start,end,fingerprint_short
0,0,51597,1800,2018-01-02,2025-02-28,fc33944bc519
1,1,51597,1800,2018-01-02,2025-02-28,fc33944bc519
2,2,51597,1800,2018-01-02,2025-02-28,fc33944bc519
3,3,51597,1800,2018-01-02,2025-02-28,fc33944bc519
4,4,51597,1800,2018-01-02,2025-02-28,fc33944bc519
5,5,51597,1800,2018-01-02,2025-02-28,fc33944bc519
6,6,51597,1800,2018-01-02,2025-02-28,fc33944bc519
7,7,51597,1800,2018-01-02,2025-02-28,fc33944bc519


Unique fingerprints in same process: 1


In [6]:
same_process_records

[{'call_id': 0,
  'pid': 51597,
  'auto_adjust': False,
  'rows': 1800,
  'start': Timestamp('2018-01-02 00:00:00'),
  'end': Timestamp('2025-02-28 00:00:00'),
  'columns': ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'],
  'fingerprint': 'fc33944bc5191ef25eff878449c787411385b8f11051d496c31c419dbf0883c0',
  'df': Price            Open       High        Low      Close  Adj Close  Volume
  Date                                                                     
  2018-01-02  53.150002  53.810001  52.410000  53.750000  40.094597  630200
  2018-01-03  54.000000  54.310001  53.619999  53.759998  40.102055  696300
  2018-01-04  54.459999  54.720001  53.980000  53.990002  40.273617  313600
  2018-01-05  54.240002  54.570000  54.060001  54.459999  40.624214  298700
  2018-01-08  54.930000  54.930000  54.480000  54.630001  40.751022  293800
  ...               ...        ...        ...        ...        ...     ...
  2025-02-24  28.750000  28.930000  28.549999  28.879999  27.731962  17

In [4]:
same_process_records = [download_once(i, auto_adjust=False) for i in range(8)]
same_process_summary = summarize_download_records(same_process_records)
display(same_process_summary[["call_id", "pid", "rows", "start", "end", "fingerprint_short"]])
print("Unique fingerprints in same process:", same_process_summary["fingerprint"].nunique())

,call_id,pid,rows,start,end,fingerprint_short
0,0,51487,1800,2018-01-02,2025-02-28,0843123606ec
1,1,51487,1800,2018-01-02,2025-02-28,0843123606ec
2,2,51487,1800,2018-01-02,2025-02-28,0843123606ec
3,3,51487,1800,2018-01-02,2025-02-28,0843123606ec
4,4,51487,1800,2018-01-02,2025-02-28,0843123606ec
5,5,51487,1800,2018-01-02,2025-02-28,0843123606ec
6,6,51487,1800,2018-01-02,2025-02-28,0843123606ec
7,7,51487,1800,2018-01-02,2025-02-28,0843123606ec


Unique fingerprints in same process: 1


In [5]:
same_process_records

[{'call_id': 0,
  'pid': 51487,
  'auto_adjust': False,
  'rows': 1800,
  'start': Timestamp('2018-01-02 00:00:00'),
  'end': Timestamp('2025-02-28 00:00:00'),
  'columns': ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'],
  'fingerprint': '0843123606ec2d05818f5b275a7e2678c349a05ca9c2a9c404197cc0549bad46',
  'df': Price            Open       High        Low      Close  Adj Close  Volume
  Date                                                                     
  2018-01-02  53.150002  53.810001  52.410000  53.750000  40.094593  630200
  2018-01-03  54.000000  54.310001  53.619999  53.759998  40.102058  696300
  2018-01-04  54.459999  54.720001  53.980000  53.990002  40.273621  313600
  2018-01-05  54.240002  54.570000  54.060001  54.459999  40.624214  298700
  2018-01-08  54.930000  54.930000  54.480000  54.630001  40.751030  293800
  ...               ...        ...        ...        ...        ...     ...
  2025-02-24  28.750000  28.930000  28.549999  28.879999  27.731962  17

From the above results we see that using the same processor will lead to identical datasets, but using different processors (by restarting the kernel between running the two cells above) will lead to different `Adj Close` values. Same processor produces identical datasets just because it is not actually downloaded again, but just loaded from the cache.

## 4. Independent Worker Processes With `auto_adjust=False`

Now we mimic joblib grid-search workers. Every call uses identical yfinance arguments. First, we keep the full yfinance output, including `Adj Close`.

In [18]:
parallel_unadjusted_records = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(
    delayed(download_once)(i, auto_adjust=False)
    for i in range(N_CALLS)
)

parallel_unadjusted_summary = summarize_download_records(parallel_unadjusted_records)
display(parallel_unadjusted_summary[["call_id", "pid", "rows", "start", "end", "columns", "fingerprint_short"]])
print("Unique full-output fingerprints:", parallel_unadjusted_summary["fingerprint"].nunique())

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Batch computation too fast (0.06787896156311035s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   8 | elapsed:    0.1s remaining:    0.2s
[Parallel(n_jobs=4)]: Done   3 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   4 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   5 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   6 out of   8 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   8 out of   8 | elapsed:    0.1s finished


,call_id,pid,rows,start,end,columns,fingerprint_short
0,0,51394,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",9d7b588b31e2
1,1,51396,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",71c8a93fbb1f
2,2,51395,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",d7f90bd63140
3,3,51393,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",8d87f07536b9
4,4,51394,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",9d7b588b31e2
5,5,51396,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",71c8a93fbb1f
6,6,51393,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",8d87f07536b9
7,7,51395,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",d7f90bd63140


Unique full-output fingerprints: 4


(MATEO:) I'm curious to see if the number of different fingerprints (ie different datasets) is equal to the number of workers. Let me test downloading very many times:

In [19]:
parallel_unadjusted_records_40 = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(
    delayed(download_once)(i, auto_adjust=False)
    # for i in range(N_CALLS)
    for i in range(40)
)

parallel_unadjusted_summary_40 = summarize_download_records(parallel_unadjusted_records_40)
display(parallel_unadjusted_summary_40[["call_id", "pid", "rows", "start", "end", "columns", "fingerprint_short"]])
print("Unique full-output fingerprints:", parallel_unadjusted_summary_40["fingerprint"].nunique())

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Batch computation too fast (0.07907319068908691s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Batch computation too fast (0.10757803916931152s.) Setting batch_size=4.
[Parallel(n_jobs=4)]: Done  12 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done  28 out of  40 | elapsed:    0.5s remaining:    0.2s
[Parallel(n_jobs=4)]: Done  40 out of  40 | elapsed:    0.5s finished


,call_id,pid,rows,start,end,columns,fingerprint_short
0,0,51394,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",9d7b588b31e2
1,1,51395,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",d7f90bd63140
2,2,51393,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",8d87f07536b9
3,3,51396,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",71c8a93fbb1f
4,4,51394,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",9d7b588b31e2
5,5,51395,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",d7f90bd63140
6,6,51393,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",8d87f07536b9
7,7,51396,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",71c8a93fbb1f
8,8,51395,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",d7f90bd63140
9,9,51395,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Adj Close, Volume]",d7f90bd63140


Unique full-output fingerprints: 4


(MATEO:)  The number of unique fingerprints coincides with the number of workers used (aka the number of different pid's (processor id)).

This makes sense of course, because when a new download call is done by a processor that already did an identical call, the result is taken from the cache. Every call by a single processor will have the same dataset.

In [6]:
parallel_unadjusted_records

[{'call_id': 0,
  'pid': 44532,
  'auto_adjust': False,
  'rows': 1800,
  'start': Timestamp('2018-01-02 00:00:00'),
  'end': Timestamp('2025-02-28 00:00:00'),
  'columns': ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'],
  'fingerprint': '29a52f0c149e2a4415acb1b7512b7dc9922bc416f301e525ed458cd81847330e',
  'df': Price            Open       High        Low      Close  Adj Close  Volume
  Date                                                                     
  2018-01-02  53.150002  53.810001  52.410000  53.750000  40.094597  630200
  2018-01-03  54.000000  54.310001  53.619999  53.759998  40.102047  696300
  2018-01-04  54.459999  54.720001  53.980000  53.990002  40.273624  313600
  2018-01-05  54.240002  54.570000  54.060001  54.459999  40.624214  298700
  2018-01-08  54.930000  54.930000  54.480000  54.630001  40.751034  293800
  ...               ...        ...        ...        ...        ...     ...
  2025-02-24  28.750000  28.930000  28.549999  28.879999  27.731962  17

The full-output fingerprints differ. The next cell measures where they differ.

In [7]:
unadjusted_diff_summary = compare_to_first(parallel_unadjusted_records)
display(unadjusted_diff_summary)

# Show the maximum difference per column against the first worker result.
base = parallel_unadjusted_records[0]["df"]
base_pid = parallel_unadjusted_records[0]["pid"]
other_pids = { record["pid"] for record in parallel_unadjusted_records[1:] if record["pid"] != base_pid }
seen_pids = set()
for record in parallel_unadjusted_records:
    if record["pid"] in other_pids and record["pid"] not in seen_pids:
        seen_pids.add(record["pid"])
        diff = (base - record["df"]).abs()
        print(f"Compared base pid {base_pid} vs pid {record['pid']}")
        display(diff.max().sort_values(ascending=False).to_frame("max_abs_diff"))

,base_pid,other_pid,index_matches,columns_match,shape(diff),shape(other),max_abs_diff,differing_cells_gt_0,differing_cells_gt_1e_5,max_pct_diff,differing_cells_pct_gt_1e_7,differing_columns,max_diff_column
0,44532,44533,True,True,"(1800, 6)","(1800, 6)",0.000015,1224,20,4.411952e-07,514,[Adj Close],Adj Close
1,44532,44534,True,True,"(1800, 6)","(1800, 6)",0.000015,1235,33,4.663830e-07,547,[Adj Close],Adj Close
2,44532,44535,True,True,"(1800, 6)","(1800, 6)",0.000015,1236,31,4.601295e-07,551,[Adj Close],Adj Close
3,44532,44532,True,True,"(1800, 6)","(1800, 6)",0.000000,0,0,0.000000e+00,0,[],None
4,44532,44532,True,True,"(1800, 6)","(1800, 6)",0.000000,0,0,0.000000e+00,0,[],None
5,44532,44533,True,True,"(1800, 6)","(1800, 6)",0.000015,1224,20,4.411952e-07,514,[Adj Close],Adj Close
6,44532,44534,True,True,"(1800, 6)","(1800, 6)",0.000015,1235,33,4.663830e-07,547,[Adj Close],Adj Close


Compared base pid 44532 vs pid 44533


,max_abs_diff
Price,
Adj Close,0.000015
Open,0.000000
High,0.000000
Low,0.000000
Close,0.000000
Volume,0.000000


Compared base pid 44532 vs pid 44534


,max_abs_diff
Price,
Adj Close,0.000015
Open,0.000000
High,0.000000
Low,0.000000
Close,0.000000
Volume,0.000000


Compared base pid 44532 vs pid 44535


,max_abs_diff
Price,
Adj Close,0.000015
Open,0.000000
High,0.000000
Low,0.000000
Close,0.000000
Volume,0.000000


**Interpretation:** with `auto_adjust=False`, the raw executable OHLCV bars match. The observed process-to-process differences are in `Adj Close`, and they are tiny, usually around `1e-5`.

In [8]:
# Now ignore Adj Close and fingerprint only executable raw OHLCV.
raw_ohlcv_columns = ["Open", "High", "Low", "Close", "Volume"]
raw_records = []
for record in parallel_unadjusted_records:
    raw_df = canonicalize_download(record["df"], columns=raw_ohlcv_columns)
    raw_records.append({
        **{k: v for k, v in record.items() if k != "df"},
        "fingerprint": dataframe_fingerprint(raw_df),
        "df": raw_df,
    })

raw_summary = summarize_download_records(raw_records)
display(raw_summary[["call_id", "pid", "rows", "columns", "fingerprint_short"]])
print("Unique raw OHLCV fingerprints:", raw_summary["fingerprint"].nunique())

,call_id,pid,rows,columns,fingerprint_short
0,0,44532,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6
1,1,44533,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6
2,2,44534,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6
3,3,44535,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6
4,4,44532,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6
5,5,44532,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6
6,6,44533,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6
7,7,44534,1800,"[Open, High, Low, Close, Adj Close, Volume]",ce1264079bc6


Unique raw OHLCV fingerprints: 1


One single unique fingerprint means that the the raw OHLCV values are identical in every download. The __difference is then only in `Adj Close`__

#### >> The `Adj_close` values are different in different downloads!!

## 5. Why `auto_adjust=True` Makes The Whole Dataset Differ



yfinance adjusts OHLC using this formula internally:

```python
ratio = Adj Close / Close
Adj Open = Open * ratio
Adj High = High * ratio
Adj Low = Low * ratio
Close = Adj Close
```

So if `Adj Close` differs by a tiny amount, then `Open`, `High`, `Low`, and `Close` all differ after `auto_adjust=True`.

In [9]:
parallel_adjusted_records = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(
    delayed(download_once)(i, auto_adjust=True)
    for i in range(N_CALLS)
)

parallel_adjusted_summary = summarize_download_records(parallel_adjusted_records)
display(parallel_adjusted_summary[["call_id", "pid", "rows", "start", "end", "columns", "fingerprint_short"]])
print("Unique adjusted-output fingerprints:", parallel_adjusted_summary["fingerprint"].nunique())

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Batch computation too fast (0.04312396049499512s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   8 | elapsed:    0.1s remaining:    0.2s
[Parallel(n_jobs=4)]: Done   3 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   4 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   5 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   6 out of   8 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   8 out of   8 | elapsed:    0.1s finished


,call_id,pid,rows,start,end,columns,fingerprint_short
0,0,44534,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",a4061bc6d2f0
1,1,44533,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",cf5220f70d88
2,2,44532,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",3d035ff0f01f
3,3,44534,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",a4061bc6d2f0
4,4,44533,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",cf5220f70d88
5,5,44535,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",429cd044ddc6
6,6,44532,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",3d035ff0f01f
7,7,44534,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",a4061bc6d2f0


Unique adjusted-output fingerprints: 4


In [10]:
adjusted_diff_summary = compare_to_first(parallel_adjusted_records)
display(adjusted_diff_summary)

base = parallel_adjusted_records[0]["df"]
base_pid = parallel_adjusted_records[0]["pid"]
other_pids = { record["pid"] for record in parallel_adjusted_records[1:] if record["pid"] != base_pid }
seen_pids = set()
for record in parallel_adjusted_records:
    if record["pid"] in other_pids and record["pid"] not in seen_pids:
        seen_pids.add(record["pid"])
        diff = (base - record["df"]).abs()
        print(f"Compared base pid {base_pid} vs pid {record['pid']}")
        display(diff.max().sort_values(ascending=False).to_frame("max_abs_diff"))

,base_pid,other_pid,index_matches,columns_match,shape(diff),shape(other),max_abs_diff,differing_cells_gt_0,differing_cells_gt_1e_5,max_pct_diff,differing_cells_pct_gt_1e_7,differing_columns,max_diff_column
0,44534,44533,True,True,"(1800, 5)","(1800, 5)",0.000015,4896,132,4.609211e-07,2020,"[Open, High, Low, Close]",High
1,44534,44532,True,True,"(1800, 5)","(1800, 5)",0.000015,4940,132,4.663828e-07,2188,"[Open, High, Low, Close]",Open
2,44534,44534,True,True,"(1800, 5)","(1800, 5)",0.000000,0,0,0.000000e+00,0,[],None
3,44534,44533,True,True,"(1800, 5)","(1800, 5)",0.000015,4896,132,4.609211e-07,2020,"[Open, High, Low, Close]",High
4,44534,44535,True,True,"(1800, 5)","(1800, 5)",0.000016,4652,132,7.328779e-07,1964,"[Open, High, Low, Close]",High
5,44534,44532,True,True,"(1800, 5)","(1800, 5)",0.000015,4940,132,4.663828e-07,2188,"[Open, High, Low, Close]",Open
6,44534,44534,True,True,"(1800, 5)","(1800, 5)",0.000000,0,0,0.000000e+00,0,[],None


Compared base pid 44534 vs pid 44533


,max_abs_diff
Price,
High,0.000015
Open,0.000015
Close,0.000015
Low,0.000015
Volume,0.000000


Compared base pid 44534 vs pid 44532


,max_abs_diff
Price,
Open,0.000015
High,0.000015
Close,0.000015
Low,0.000015
Volume,0.000000


Compared base pid 44534 vs pid 44535


,max_abs_diff
Price,
High,0.000016
Open,0.000015
Close,0.000015
Low,0.000015
Volume,0.000000


## 6. Deterministic Download Wrapper For Executable Prices



For this strategy, the backtest executes at market `Open`. Those are raw executable prices, not adjusted prices. The deterministic wrapper below downloads with `auto_adjust=False`, disables yfinance's internal ticker threading, and keeps only raw `Open/High/Low/Close/Volume`. In the process-parallel test, this produces identical fingerprints.

In [11]:
def download_executable_ohlcv(call_id):
    """Download deterministic executable OHLCV for this test.

    This deliberately discards Adj Close. The raw OHLCV bars are the exchange-like
    prices used for execution and were stable across independent worker processes
    in the tests above.
    """
    return download_once(call_id, auto_adjust=False, columns=["Open", "High", "Low", "Close", "Volume"])

executable_records = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(
    delayed(download_executable_ohlcv)(i)
    for i in range(N_CALLS)
)

executable_summary = summarize_download_records(executable_records)
display(executable_summary[["call_id", "pid", "rows", "start", "end", "columns", "fingerprint_short"]])
print("Unique executable-OHLCV fingerprints:", executable_summary["fingerprint"].nunique())

assert executable_summary["fingerprint"].nunique() == 1, "Executable OHLCV downloads were not deterministic."

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Batch computation too fast (0.04221916198730469s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   8 | elapsed:    0.0s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   3 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   4 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   5 out of   8 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=4)]: Done   6 out of   8 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   8 out of   8 | elapsed:    0.1s finished


,call_id,pid,rows,start,end,columns,fingerprint_short
0,0,44533,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
1,1,44535,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
2,2,44532,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
3,3,44534,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
4,4,44533,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
5,5,44535,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
6,6,44532,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
7,7,44534,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6


Unique executable-OHLCV fingerprints: 1


In [12]:
# executable_records = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(
executable_records = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(download_executable_ohlcv)(i)
    # for i in range(N_CALLS)
    for i in range(40)
)

executable_summary = summarize_download_records(executable_records)
display(executable_summary[["call_id", "pid", "rows", "start", "end", "columns", "fingerprint_short"]])
print("Unique executable-OHLCV fingerprints:", executable_summary["fingerprint"].nunique())

assert executable_summary["fingerprint"].nunique() == 1, "Executable OHLCV downloads were not deterministic."

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done  30 out of  40 | elapsed:    2.7s remaining:    0.9s
[Parallel(n_jobs=-1)]: Done  35 out of  40 | elapsed:    2.7s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done  40 out of  40 | elapsed:    2.8s finished


,call_id,pid,rows,start,end,columns,fingerprint_short
0,0,44542,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
1,1,44543,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
2,2,44544,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
3,3,44545,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
4,4,44546,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
5,5,44547,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
6,6,44548,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
7,7,44549,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
8,8,44542,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6
9,9,44546,1800,2018-01-02,2025-02-28,"[Open, High, Low, Close, Volume]",ce1264079bc6


Unique executable-OHLCV fingerprints: 1


In [13]:
unique_pids = { record["pid"] for record in executable_records }
display(pd.DataFrame({"pid": list(unique_pids)}))
print("Unique PIDs for executable OHLCV downloads:", len(unique_pids))
print("Number of fingerprints of OHLCV downloads:", executable_summary["fingerprint"].nunique())

,pid
0,44544
1,44545
2,44546
3,44547
4,44548
5,44549
6,44542
7,44543


Unique PIDs for executable OHLCV downloads: 8
Number of fingerprints of OHLCV downloads: 1


## 7. Recommended Change To The ML Predictor



The safest fix is to stop using adjusted OHLC as the model/backtest price source. In `ML_predictor.py`, prefer:

```python
df = yf.download(
    symbol,
    start=start_date_download,
    end=end_date,
    auto_adjust=False,
    progress=self.verbose,
    threads=False,
    timeout=30,
)

df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
```

This does two things:

1. It avoids yfinance's tiny process-to-process `Adj Close` variations.
2. It uses raw executable `Open` prices for a strategy that trades at the open.

If you later want adjusted data for some feature engineering, compute or freeze that separately. Do not let live adjusted yfinance data be regenerated independently inside every parameter-search worker.

## 8. Conclusion



In this environment, the problem is not the model, indicators, backtester, `parameter_grid_search`, or yfinance `threads=True` alone. The reproducible trigger is:

- independent worker processes,
- same yfinance arguments,
- yfinance/Yahoo `Adj Close` values that vary slightly across processes,
- `auto_adjust=True`, which propagates those small `Adj Close` differences into all OHLC columns.

The practical mend is to download raw OHLCV with `auto_adjust=False`, discard `Adj Close`, and use those raw prices consistently for execution-based backtests.

#### (MATEO:) I tested the whole thing changing `threads=True` and exactly the same conclusions are drawn, which means the problem is only with `auto_adjusted=True`, while `threads=True` does not cause the datasets to differ at all, so it can be safely kept for time-efficiency.